# 🧠 BraTS 2024 nnU-Net Training Pipeline (Upgraded)

This notebook is redesigned to be:
- Reliable
- Modular
- Debuggable
- Interview-ready


In [1]:
# Uncomment once if needed:
# !pip install -q nnunetv2 monai nibabel pandas tqdm

import os, re, json, time, shutil, random, subprocess, math
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import torch
from tqdm.auto import tqdm
from monai.metrics import compute_hausdorff_distance

from monai.data import Dataset, DataLoader

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths (UNCHANGED as requested) ──
DATA_ROOT            = Path(r"C:\Users\Admin\Desktop\meghana\Brats2024\BraTS2024-BraTS-GLI-TrainingData\training_data1_v2")
NNUNET_RAW           = Path("./nnunet_raw")
NNUNET_PREPROCESSED  = Path("./nnunet_preprocessed")
NNUNET_RESULTS       = Path("./nnunet_results")
DATASET_ID           = 500
DATASET_NAME         = f"Dataset{DATASET_ID:03d}_BraTSGlioma"

# ── Environment Variables ──
os.environ["nnUNet_raw"]          = str(NNUNET_RAW.resolve())
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED.resolve())
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS.resolve())

# ── CPU Optimization (IMPORTANT for your Xeon) ──
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"

# ── Create Required Folders ──
for p in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

# ── Basic Checks ──
assert DATA_ROOT.exists(), f"❌ DATA_ROOT not found: {DATA_ROOT}"

cases = sorted([p for p in DATA_ROOT.glob("*") if p.is_dir()])
print(f"📦 Total cases found: {len(cases)}")
assert len(cases) > 0, "❌ No cases found in dataset"

# Preview one case
sample_case = cases[0]
print(f"\n🔍 Sample case: {sample_case.name}")
print("Files:", [f.name for f in sample_case.glob("*")])

# ── Device Setup ──
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

print("\n🖥️ Device:", DEVICE)

if DEVICE == "cuda":
    print("🚀 GPU:", torch.cuda.get_device_name(0))
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Running on CPU — training will be very slow")

# ── Quick GPU Check ──
def check_gpu():
    if torch.cuda.is_available():
        print("✅ CUDA is working properly")
    else:
        print("❌ CUDA not available")

check_gpu()

c:\Users\Admin\Desktop\meghana\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Total cases found: 1350

🔍 Sample case: BraTS-GLI-00005-100
Files: ['BraTS-GLI-00005-100-seg.nii.gz', 'BraTS-GLI-00005-100-t1c.nii', 'BraTS-GLI-00005-100-t1c.nii.gz', 'BraTS-GLI-00005-100-t1n.nii.gz', 'BraTS-GLI-00005-100-t2f.nii.gz', 'BraTS-GLI-00005-100-t2w.nii.gz']

🖥️ Device: cuda
🚀 GPU: NVIDIA RTX A6000
💾 VRAM: 51.5 GB
✅ CUDA is working properly


In [2]:
# =========================
# STEP 2: DISCOVERY + CONVERSION + LABEL FIX (FINAL)
# =========================

def find_case_files(case_dir: Path):
    nii_files = sorted(case_dir.glob("*.nii.gz"))

    def pick(patterns):
        for p in patterns:
            for f in nii_files:
                if re.search(p, f.name.lower()):
                    return f
        return None

    t1    = pick([r"-t1n\.nii\.gz$", r"-t1\.nii\.gz$",    r"_t1n\.nii\.gz$", r"_t1\.nii\.gz$"])
    t1ce  = pick([r"-t1c\.nii\.gz$", r"-t1ce\.nii\.gz$",  r"_t1c\.nii\.gz$", r"_t1ce\.nii\.gz$"])
    t2    = pick([r"-t2w\.nii\.gz$", r"-t2\.nii\.gz$",    r"_t2w\.nii\.gz$", r"_t2\.nii\.gz$"])
    flair = pick([r"-t2f\.nii\.gz$", r"-flair\.nii\.gz$", r"_t2f\.nii\.gz$", r"_flair\.nii\.gz$"])
    seg   = pick([r"-seg\.nii\.gz$", r"_seg\.nii\.gz$",   r"-mask\.nii\.gz$"])

    if all(x is not None for x in [t1, t1ce, t2, flair, seg]):
        return {"case_id": case_dir.name, "t1": t1, "t1ce": t1ce, "t2": t2, "flair": flair, "seg": seg}
    return None


# ── Discover cases ──
case_dirs = [p for p in sorted(DATA_ROOT.iterdir()) if p.is_dir()]
cases = []

for c in tqdm(case_dirs, desc="Scanning cases"):
    item = find_case_files(c)
    if item:
        cases.append(item)

print(f"✅ Discovered {len(cases)} valid cases")
assert len(cases) > 0, "❌ No valid cases found"


# ── Prepare nnU-Net folders ──
out_ds   = NNUNET_RAW / DATASET_NAME
imagesTr = out_ds / "imagesTr"
labelsTr = out_ds / "labelsTr"

# Clean old data (IMPORTANT)
if imagesTr.exists():
    shutil.rmtree(imagesTr)
if labelsTr.exists():
    shutil.rmtree(labelsTr)

imagesTr.mkdir(parents=True, exist_ok=True)
labelsTr.mkdir(parents=True, exist_ok=True)


# ── Convert + Fix Labels INLINE ──
print("🚀 Converting and fixing labels...")

for i, case in enumerate(tqdm(cases, desc="Processing")):
    case_id = f"BraTS_{i:04d}"

    # nnU-Net modality order (IMPORTANT)
    shutil.copy(case["flair"], imagesTr / f"{case_id}_0000.nii.gz")
    shutil.copy(case["t1"],    imagesTr / f"{case_id}_0001.nii.gz")
    shutil.copy(case["t1ce"],  imagesTr / f"{case_id}_0002.nii.gz")
    shutil.copy(case["t2"],    imagesTr / f"{case_id}_0003.nii.gz")

    # Load + remap labels
    img  = nib.load(str(case["seg"]))
    data = img.get_fdata().astype(np.int16)

    if 4 in np.unique(data):
        data[data == 4] = 3

    nib.save(
        nib.Nifti1Image(data, img.affine, img.header),
        str(labelsTr / f"{case_id}.nii.gz")
    )


# ── dataset.json ──
dataset_json = {
    "channel_names": {
        "0": "FLAIR",
        "1": "T1",
        "2": "T1ce",
        "3": "T2"
    },
    "labels": {
        "background": 0,
        "NCR_NET": 1,
        "edema": 2,
        "ET": 3
    },
    "numTraining": len(cases),
    "file_ending": ".nii.gz"
}

with open(out_ds / "dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=2)


# ── Final sanity check ──
num_cases = len(cases)
num_images = len(list(imagesTr.glob("*_0000.nii.gz")))
num_labels = len(list(labelsTr.glob("*.nii.gz")))

print("\n📊 FINAL CHECK")
print(f"Cases:   {num_cases}")
print(f"Images:  {num_images}")
print(f"Labels:  {num_labels}")

assert num_images == num_cases, "❌ Image count mismatch"
assert num_labels == num_cases, "❌ Label count mismatch"

print("\n✅ Dataset READY for nnU-Net")

Scanning cases: 100%|██████████| 1350/1350 [00:00<00:00, 3149.91it/s]


✅ Discovered 1350 valid cases
🚀 Converting and fixing labels...


Processing: 100%|██████████| 1350/1350 [15:20<00:00,  1.47it/s]


📊 FINAL CHECK
Cases:   1350
Images:  1350
Labels:  1350

✅ Dataset READY for nnU-Net


In [3]:
!nnUNetv2_plan_and_preprocess -d 500 --no_pp

Fingerprint extraction...
Dataset500_BraTSGlioma
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [142. 175. 136.], 3d_lowres: [142, 175, 136]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([175., 136.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resampl

In [4]:
!nnUNetv2_plan_and_preprocess -d 500 -np 12 --clean --verbose

Fingerprint extraction...
Dataset500_BraTSGlioma
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [142. 175. 136.], 3d_lowres: [142, 175, 136]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([175., 136.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True]

In [5]:
!nnUNetv2_plan_and_preprocess -d 500 -c 3d_fullres -np 8 --verbose --clean

Fingerprint extraction...
Dataset500_BraTSGlioma
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [142. 175. 136.], 3d_lowres: [142, 175, 136]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([175., 136.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True]

In [6]:
!nnUNetv2_train Dataset500_BraTSGlioma 3d_fullres 0 -device cuda

^C
